In [0]:
# 1. database name
storage_account_name = "dlspl21databricks"
container_name = "konstyantyn"
secret_scope_name = "konstyantyn_key_secret_scope"

# 2. safe secret keys stored in Azure Key Vault
client_id = dbutils.secrets.get(scope=secret_scope_name, key="client-id")
client_secret = dbutils.secrets.get(scope=secret_scope_name, key="client-secret")
tenant_id = dbutils.secrets.get(scope=secret_scope_name, key="tenant-id")

# 3. Service Principal configuration for Azure Databricks
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# 4. local mount point for Databricks (DBFS)
mount_point = f"/mnt/{container_name}_legacy"

# 5. test for mount point and container
if not any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
    print(f"Mounting container '{container_name}' to '{mount_point}'...")
    dbutils.fs.mount(
        source=f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/",
        mount_point=mount_point,
        extra_configs=configs
    )
    print("Mount successful!")
else:
    print(f"Mount point {mount_point} already exists.")

# 6. test for data
display(dbutils.fs.ls(mount_point))

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-7289020028915629>, line 7
      4 secret_scope_name = "konstyantyn_key_secret_scope"
      6 # 2. safe secret keys stored in Azure Key Vault
----> 7 client_id = dbutils.secrets.get(scope=secret_scope_name, key="client-id")
      8 client_secret = dbutils.secrets.get(scope=secret_scope_name, key="client-secret")
      9 tenant_id = dbutils.secrets.get(scope=secret_scope_name, key="tenant-id")

File /databricks/python_shell/lib/dbruntime/dbutils.py:376, in DBUtils.SecretsHandler.get(self, scope, catalog, schema, key, *posArgs)
    373     scope = argsToPass[0]
    374     key = argsToPass[1]
    375     return self.entry_point.getDbutils().preview().secret(
--> 376     ).get(  # type: ignore[attr-defined]
    377         scope, key)
    378 else:
    379     catalog = argsToPass[0]

File /databricks/spark/python/lib/py4j-0.1